# Clothes segmentation: does data augmentation help?

Trains SegFormer-B2 for 5 epochs **without** data augmentation and compares it with the round 1 run `r1_segformer_b2`, which used augmentation and is identical otherwise (same data, seed, batch size and 12-epoch schedule).

SegFormer-B2 is used because it fits a 16 GB T4 at batch size 16. Expect roughly 2 to 3 hours on a T4. If the session ends, run all cells again: training continues from the last checkpoint.

## 1. Settings

In [ ]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "19"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 280        # stop cleanly and save before a 5-hour session ends

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements and prepare the dataset

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

## 5. Train without augmentation and compare

In [ ]:
RUN, BASE = "abl_segformer_b2_noaug", "r1_segformer_b2"

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {RUN} --model segformer_b2 --height {HEIGHT} --width {WIDTH} --epochs 12 --stop-after-epoch 5 --no-augment --time-budget-min {budget}

import pandas as pd
assert os.path.exists(f"{DRIVE_ROOT}/runs/{RUN}/val_log.csv"), "No epoch has finished yet. Read the training output above for the reason."
base = pd.read_csv(f"{DRIVE_ROOT}/runs/{BASE}/val_log.csv").set_index("epoch")
noaug = pd.read_csv(f"{DRIVE_ROOT}/runs/{RUN}/val_log.csv").set_index("epoch")
print("\ngarment mIoU and validation loss per epoch")
print(pd.DataFrame({"with augmentation": base["garment_miou"], "without": noaug["garment_miou"],
                    "val loss with": base["val_loss"], "val loss without": noaug["val_loss"]}).dropna().round(4).to_string())
train_loss = lambda name: pd.read_csv(f"{DRIVE_ROOT}/runs/{name}/train_log.csv").groupby("epoch")["loss"].mean()
print("\nmean training loss per epoch (a low training loss with a high validation loss means overfitting)")
print(pd.DataFrame({"with augmentation": train_loss(BASE), "without": train_loss(RUN)}).dropna().round(4).to_string())

## What to send back

The whole text output of cell 5.